# RAG Retrieval Strategy Experimentation
This notebook evaluates three retrieval approaches on the domain corpus:
1. **Lexical Keyword matching (BM25)**
2. **Semantic Dense Vector search (Cosine Similarity via all-MiniLM-L6-v2)**
3. **Hybrid Search (BM25 + Semantic fused via Reciprocal Rank Fusion - RRF)**

In [ ]:
import sys
sys.path.append('..')
from src.document_processor import DocumentProcessor
from src.retriever import Retriever
import numpy as np
print("Retrieval classes loaded successfully!")

### Load and Index Chunks
We use the Medium chunk size configuration (300 tokens) to process the corpus and build the Retriever indexes.

In [ ]:
processor = DocumentProcessor()
chunks = processor.process_directory("../data", chunk_size=300, chunk_overlap=30)
retriever = Retriever(chunks)
print(f"Successfully indexed {len(chunks)} chunks in BM25 and Vector databases!")

### Compare Retrieval Top-K Results
Let's test search queries to compare lexical matching vs conceptual similarity.

In [ ]:
query = "What medication is first line for a patient with protein in their urine or CKD?"

print("=== BM25 KEYWORD SEARCH ===")
bm25_res = retriever.retrieve(query, method="bm25", top_k=2)
for c, s in bm25_res:
    print(f"Score: {s:.3f} | {c['chunk_id']} | Text: {c['text'][:150]}...")
    
print("\n=== SEMANTIC VECTOR SEARCH ===")
sem_res = retriever.retrieve(query, method="semantic", top_k=2)
for c, s in sem_res:
    print(f"Score: {s:.3f} | {c['chunk_id']} | Text: {c['text'][:150]}...")
    
print("\n=== HYBRID RRF SEARCH ===")
hyb_res = retriever.retrieve(query, method="hybrid", top_k=2)
for c, s in hyb_res:
    print(f"RRF Score: {s:.5f} | {c['chunk_id']} | Text: {c['text'][:150]}...")

### Observations
- **BM25** matches specific keywords like 'CKD', but might rank general sentences high if they happen to contain the words.
- **Semantic search** catches concepts like 'protein in urine' (matching the clinical term 'albuminuria' in our text), showing strong semantic understanding.
- **Hybrid search (RRF)** effectively bubbles up chunks that contain both precise terminology and conceptual proximity, offering the best overall results.